<a href="https://colab.research.google.com/github/MaydaCaba/Lab2-blockchain/blob/main/Lab2_Blockchain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Laboratorio 2: Bloque + Cadena + validación

In [1]:
import hashlib
import struct
from time import time

def calcular_raiz_merkle(datos):
  #calculo de doble SHA-256 para obtener raiz de merckle de 32 bytes
  datos_bytes = str(datos).encode('utf-8')
  primera = hashlib.sha256(datos_bytes).digest()
  return hashlib.sha256(primera).hexdigest()


def calcular_hash(bloque):
  #calculo de hash real--bitcoin
  cabecera = (
      struct.pack('<i', bloque['version'])
      + bytes.fromhex(bloque["hash_previo"])[::-1]
      + bytes.fromhex(bloque["raiz_merkle"])[::-1]
      + struct.pack("<III", bloque["tiempo"], bloque["bits"], bloque["nonce"])
      )
  primera_huella = hashlib.sha256(cabecera).digest()
  return hashlib.sha256(primera_huella).digest()[::-1].hex()

def crear_bloque(version, hash_previo, datos, tiempo=None, bits=0x1d00ffff, nonce=0):
  #diccionario de bloques datos y hash calculado
  bloque = {
      "version": version,
      "hash_previo": hash_previo,
      "raiz_merkle": calcular_raiz_merkle(datos),
      "tiempo": tiempo if tiempo is not None else int(time()),
      "bits": bits,
      "nonce": nonce,
      "datos": datos
      }
  # guardamos el hash resultante en el bloque
  bloque["hash"] = calcular_hash(bloque)
  return bloque

class CadenaBloques:

  #administra la cadena de bloques y valida su integridad criptografica
  def __init__(self):
    #inicio de la cadena con bloque Genesis (bloque 0)
    genesis = {
        "version":1,
        "hash_previo": "00" * 32,
        "raiz_merkle": calcular_raiz_merkle("bloque genesis"),
        "tiempo": 1231006505,
        "bits": 0x1d00ffff,
        "nonce": 2083236893,
        "datos": "bloque genesis"
        }
    genesis["hash"] = calcular_hash(genesis)
    self.cadena = [genesis]

  def agregar_bloque(self, datos, nonce=0):
    #Bloque 1 enlazado al bloque anterior
    bloque_anterior = self.cadena[-1]
    nuevo_bloque = crear_bloque(
        version=1,
        hash_previo=bloque_anterior["hash"],
        datos=datos,
        nonce=nonce
        )
    self.cadena.append(nuevo_bloque)

  def es_cadena_valida(self):
    #verificacion la intgridad de toda la cadena
    for i in range(1, len(self.cadena)):
      actual = self.cadena[i]
      anterior = self.cadena[i - 1]

      #1-verificando la integridad de los datos mediante la raiz de Merkle
      if actual["raiz_merkle"] != calcular_raiz_merkle(actual["datos"]):
        return False

      #2-verificando la integridad de la cabecera mediante doble SHA-256
      if actual["hash"] != calcular_hash(actual):
        return False

      #3-verificando el enlace criptografico con el bloque anterior
      if actual["hash_previo"] != anterior["hash"]:
        return False

    return True

In [5]:
from pprint import pprint
import unittest

#---1 imprimir cadenas
mi_cadena = CadenaBloques()
mi_cadena.agregar_bloque("Sami envia 100 BTC a Josu", nonce=12345)
mi_cadena.agregar_bloque("Josu envia 50 BTC a Jes", nonce=67890)
mi_cadena.agregar_bloque("Jes envia 200 BTC a May", nonce=13579)

print("¿La cadena es valida?:", mi_cadena.es_cadena_valida())

pprint("===BLOQUES EN LA CADENA===")
for i, b in enumerate(mi_cadena.cadena):
  print(f"\nBloque #{i} -> Hash: {b['hash']}")
  pprint(b, sort_dicts=False)

#---2 automatizacion de validacion y manipulacion
class PruebasBlockchain(unittest.TestCase):
  def setUp(self):
    #cadena limpia antes de cada prueba
    self.bc = CadenaBloques()
    self.bc.agregar_bloque("Sami envia 100 BTC a Josu", nonce=12345)
    self.bc.agregar_bloque("Josu envia 50 BTC a Jes", nonce=67890)

  def test_1_cadena_valida(self):
    #cadena sin alteraciones debe ser valida (TRUE)
    self.assertTrue(self.bc.es_cadena_valida())

  def test_2_manipulacion_datos(self):
    #alterando bloque 1 sin reclacular nada
    self.bc.cadena[1]["datos"] = "Sami envia 9999 BTC a Josu"
    self.assertFalse(self.bc.es_cadena_valida())

  def test_3_manipulacion_campo_cabecera_sin_recalcular(self):
    #alteramos el nonce en la cabecera del bloque 1 sin recalcular su hash
    self.bc.cadena[1]["nonce"] = 999999999
    self.assertFalse(self.bc.es_cadena_valida())

  def test_4_manipulacion_y_recalculo_hash(self):
    #alteramos los datos, actualizamos raiz de Merkle reclaculamos su hash
    self.bc.cadena[1]["datos"] = "Sami envia 9999 BTC a Josu"
    self.bc.cadena[1]["raiz_merkle"] = calcular_raiz_merkle(self.bc.cadena[1]["datos"])
    self.bc.cadena[1]["hash"] = calcular_hash(self.bc.cadena[1])
    #por fallas del hash previo del bloque 2
    self.assertFalse(self.bc.es_cadena_valida())


print("\n== RESULTADOS DE LAS PRUEBAS DE MANIPULACION==")
unittest.main(argv=[''], verbosity=2, exit=False)

test_1_cadena_valida (__main__.PruebasBlockchain.test_1_cadena_valida) ... ok
test_2_manipulacion_datos (__main__.PruebasBlockchain.test_2_manipulacion_datos) ... ok
test_3_manipulacion_campo_cabecera_sin_recalcular (__main__.PruebasBlockchain.test_3_manipulacion_campo_cabecera_sin_recalcular) ... ok
test_4_manipulacion_y_recalculo_hash (__main__.PruebasBlockchain.test_4_manipulacion_y_recalculo_hash) ... ok

----------------------------------------------------------------------
Ran 4 tests in 0.009s

OK


¿La cadena es valida?: True
'===BLOQUES EN LA CADENA==='

Bloque #0 -> Hash: 1b96b32ee078491b9c86310303fbeb1d9d7ff56d62924c06e653307152e7b001
{'version': 1,
 'hash_previo': '0000000000000000000000000000000000000000000000000000000000000000',
 'raiz_merkle': 'acb308aa9d54548261893786199b84f614f9f2a6ea6e07a96cf56f451c911768',
 'tiempo': 1231006505,
 'bits': 486604799,
 'nonce': 2083236893,
 'datos': 'bloque genesis',
 'hash': '1b96b32ee078491b9c86310303fbeb1d9d7ff56d62924c06e653307152e7b001'}

Bloque #1 -> Hash: d11a4dae6b3b317bd982bc88935eb202f39e905429e6f317969259b451370dbc
{'version': 1,
 'hash_previo': '1b96b32ee078491b9c86310303fbeb1d9d7ff56d62924c06e653307152e7b001',
 'raiz_merkle': '788805bbcbde7b3415790659c116e331cb45591b43759ec41efbd7078bf02911',
 'tiempo': 1794764480,
 'bits': 486604799,
 'nonce': 12345,
 'datos': 'Sami envia 100 BTC a Josu',
 'hash': 'd11a4dae6b3b317bd982bc88935eb202f39e905429e6f317969259b451370dbc'}

Bloque #2 -> Hash: 58d7b547ec237acec832456a83ed04e45e584b491